# Fakeddit — Optuna Hyperparameter Tuning

Standalone notebook extracted from `misinformation_ver4-1_full_IP_variant.ipynb`. It loads and cleans the Fakeddit data, runs **one** of the five Optuna studies (selected via the `TRIAL_TO_RUN` variable), then shows:

1. the **confusion matrix** of the best model on the held-out test set,
2. the **validation score over trials** (optimization history).

Both plots are saved to `artifacts/plots/`, suffixed with the trial name, so the studies can be run one after another and compared.

In [ ]:
import numpy as np
import polars as pl
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from pathlib import Path
from typing import Self

from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

## Data Loading

In [ ]:
all_data extraction in-cloud
SAMPLES_DIR = Path("/kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples")

all_data: pl.DataFrame = pl.concat([
    pl.read_csv(SAMPLES_DIR / filename, separator="\t") for filename in SAMPLES_DIR.iterdir()
]).with_columns(
    pl.from_epoch("created_utc", time_unit="s").alias("created_date")
)


In [ ]:
# Same cleaning as the full notebook: dedupe on (title, image_url), keep text + 6-way label.
trimmed_data = all_data.select(
    pl.col("author"),
    pl.col("created_utc"),
    pl.col("clean_title"),
    pl.col("score"),
    pl.col("2_way_label"),
    pl.col("6_way_label"),
    pl.col("num_comments"),
    pl.col("upvote_ratio"),
    pl.col("image_url"),
).unique(
    subset=["clean_title", "image_url"],
    keep="first"
).drop("image_url")

model_data = (
    trimmed_data
    .select(["clean_title", "6_way_label"])
    .drop_nulls()
)

X = model_data["clean_title"]
y = model_data["6_way_label"]

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train size:", len(X_train_multi))
print("Test size:", len(X_test_multi))

In [ ]:
# Transformer for embeddings (needed by the embedding-based trials).

class TitleEmbedder(TransformerMixin, BaseEstimator):
    """
    A custom scikit-learn transformer that wraps HuggingFace SentenceTransformers.
    This allows us to seamlessly integrate deep learning embeddings into standard ML pipelines.
    """
    def __init__(self, model_name: str, *, batch_size: int = 256) -> None:
        self._model_name = model_name
        self._batch_size = batch_size
        self._model = None # Initialized during fit

    def fit(self, X: np.ndarray, y: None = None) -> Self:
        # Load the pre-trained transformer model
        self._model = SentenceTransformer(self._model_name)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        # Convert text into dense vector embeddings
        return self._model.encode(
            list(X),
            batch_size=self._batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

## Hyperparameter Tuning with Optuna

Five studies are available: the TF-IDF baseline plus a 2×2 grid of embedder × classifier. Pick one with `TRIAL_TO_RUN`:

| `TRIAL_TO_RUN` | Model | Tuned hyperparameters |
|---|---|---|
| `"baseline"` | TF-IDF + logistic regression | `max_df`, `min_df`, `c_param` |
| `"paraphrase_logreg"` | paraphrase-MiniLM-L3-v2 embeddings + logistic regression | `c_param` |
| `"paraphrase_lgbm"` | paraphrase-MiniLM-L3-v2 embeddings + LightGBM | `n_estimators`, `learning_rate`, `num_leaves`, `max_depth` |
| `"minilm_logreg"` | all-MiniLM-L12-v2 embeddings + logistic regression | `c_param` |
| `"minilm_lgbm"` | all-MiniLM-L12-v2 embeddings + LightGBM | `n_estimators`, `learning_rate`, `num_leaves`, `max_depth` |

Both embedders share the exact same classifier search spaces and data splits, so the comparison between them is fair.

`use_quick_test = True` keeps the run laptop-friendly (500 train / 100 test samples, 10 trials).

In [ ]:
# Which Optuna study to run:
#   "baseline"          - TF-IDF + logistic regression
#   "paraphrase_logreg" - paraphrase-MiniLM-L3-v2 embeddings + logistic regression
#   "paraphrase_lgbm"   - paraphrase-MiniLM-L3-v2 embeddings + LightGBM
#   "minilm_logreg"     - all-MiniLM-L12-v2 embeddings + logistic regression
#   "minilm_lgbm"       - all-MiniLM-L12-v2 embeddings + LightGBM
TRIAL_TO_RUN = "paraphrase_logreg"

# Sentence-transformer used by each embedding-based trial.
TRIAL_EMBEDDERS = {
    "paraphrase_logreg": "paraphrase-MiniLM-L3-v2",
    "paraphrase_lgbm": "paraphrase-MiniLM-L3-v2",
    "minilm_logreg": "all-MiniLM-L12-v2",
    "minilm_lgbm": "all-MiniLM-L12-v2",
}

# Set to False for the final full run.
use_quick_test = True

# Where the result plots get saved.
PLOTS_DIR = Path("artifacts/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if use_quick_test:
    print("Running in quick test mode with reduced data and trials.")
    x_train_opt = X_train_multi[:500]
    y_train_opt = y_train_multi[:500]
    x_test_opt = X_test_multi[:100]
    y_test_opt = y_test_multi[:100]
    number_of_trials = 10
else:
    print("Running full optimization mode.")
    x_train_opt = X_train_multi
    y_train_opt = y_train_multi
    x_test_opt = X_test_multi
    y_test_opt = y_test_multi
    number_of_trials = 20

# Create an internal validation set for Optuna to avoid touching the final test set.
x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_opt, y_train_opt, test_size=0.15, random_state=42, stratify=y_train_opt
)

In [ ]:
# Precompute embeddings once so the trials do not re-encode the text every time.
# Only needed for the embedding-based studies; the embedder depends on the selected trial.
if TRIAL_TO_RUN in TRIAL_EMBEDDERS:
    embedder_name = TRIAL_EMBEDDERS[TRIAL_TO_RUN]
    print(f"Precomputing {embedder_name} embeddings to save time during trials.")
    embedder = TitleEmbedder(embedder_name, batch_size=256)
    x_tr_emb = embedder.fit_transform(x_tr)
    x_val_emb = embedder.transform(x_val)
    x_test_emb = embedder.transform(x_test_opt)

In [ ]:
# Objective functions. The embedding-based objectives are shared by both embedders
# (identical search spaces and splits, so the embedder comparison stays fair) —
# they simply consume whatever embeddings were precomputed above.

def objective_baseline(trial):
    # Optuna suggests hyperparameters for TF-IDF.
    max_df = trial.suggest_float("max_df", 0.7, 1.0)
    min_df = trial.suggest_int("min_df", 1, 5)

    # Optuna suggests hyperparameters for logistic regression.
    c_param = trial.suggest_float("c_param", 1e-3, 10.0, log=True)

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=30000, max_df=max_df, min_df=min_df, lowercase=True)),
        ("logreg", LogisticRegression(C=c_param, max_iter=1000, class_weight="balanced", random_state=42)),
    ])

    pipeline.fit(x_tr, y_tr)
    y_pred_val = pipeline.predict(x_val)
    return f1_score(y_val, y_pred_val, average="macro")


def objective_emb_logreg(trial):
    c_param = trial.suggest_float("c_param", 1e-3, 10.0, log=True)
    model = LogisticRegression(C=c_param, max_iter=1000, class_weight="balanced", random_state=42)
    model.fit(x_tr_emb, y_tr)
    y_pred_val = model.predict(x_val_emb)
    return f1_score(y_val, y_pred_val, average="macro")


def objective_emb_lgbm(trial):
    params = {
        "objective": "multiclass", "num_class": 6, "class_weight": "balanced",
        "random_state": 42, "n_jobs": -1, "verbose": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 80),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(
        x_tr_emb, y_tr,
        eval_set=[(x_val_emb, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)],
    )
    y_pred_val = model.predict(x_val_emb)
    return f1_score(y_val, y_pred_val, average="macro")


objectives = {
    "baseline": objective_baseline,
    "paraphrase_logreg": objective_emb_logreg,
    "paraphrase_lgbm": objective_emb_lgbm,
    "minilm_logreg": objective_emb_logreg,
    "minilm_lgbm": objective_emb_lgbm,
}

print(f"Optimizing {TRIAL_TO_RUN} with {number_of_trials} trials.")
study = optuna.create_study(direction="maximize")
study.optimize(objectives[TRIAL_TO_RUN], n_trials=number_of_trials)

print(f"\nBest validation f1 macro: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# Refit the best model with the tuned hyperparameters and evaluate it on the held-out test set.
if TRIAL_TO_RUN == "baseline":
    best_model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=30000,
                                  max_df=study.best_params["max_df"],
                                  min_df=study.best_params["min_df"], lowercase=True)),
        ("logreg", LogisticRegression(C=study.best_params["c_param"], max_iter=1000, class_weight="balanced", random_state=42)),
    ])
    best_model.fit(x_train_opt, y_train_opt)
    test_features = x_test_opt
elif TRIAL_TO_RUN.endswith("_logreg"):
    best_model = LogisticRegression(C=study.best_params["c_param"], max_iter=1000, class_weight="balanced", random_state=42)
    best_model.fit(x_tr_emb, y_tr)
    test_features = x_test_emb
else:  # *_lgbm
    lgbm_params = study.best_params.copy()
    lgbm_params.update({
        "objective": "multiclass", "num_class": 6, "class_weight": "balanced",
        "random_state": 42, "n_jobs": -1, "verbose": -1,
    })
    best_model = lgb.LGBMClassifier(**lgbm_params)
    best_model.fit(x_tr_emb, y_tr)
    test_features = x_test_emb

y_test_pred = best_model.predict(test_features)
f1_test = f1_score(y_test_opt, y_test_pred, average="macro")
print(f"Test f1 macro for the best {TRIAL_TO_RUN} model: {f1_test:.4f}\n")

label_names = ["True", "Satire / parody", "False connection", "Imposter content", "Manipulated", "Misleading"]

# Map only the classes actually present, so small quick-test runs do not crash the report.
seen_labels = sorted(set(np.asarray(y_test_opt)) | set(np.asarray(y_test_pred)))
seen_names = [label_names[label] for label in seen_labels]
print(classification_report(y_test_opt, y_test_pred, labels=seen_labels, target_names=seen_names, digits=4, zero_division=0))

## Results

Two views of the optimization run: the confusion matrix of the best model on the held-out test set, and the validation score of every trial (with the best-so-far envelope).

Both plots are also saved to `artifacts/plots/` as PNGs, suffixed with the trial name (e.g. `confusion_matrix_baseline.png`, `optuna_history_baseline.png`), so running different `TRIAL_TO_RUN` values does not overwrite earlier results.

In [ ]:
# Plot 1: confusion matrix for the best model on the held-out test set.
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_opt, y_test_pred,
    labels=seen_labels,
    display_labels=seen_names,
    cmap="Blues",
    ax=ax,
    xticks_rotation=45,
)
ax.set_title(f"Confusion matrix for the best {TRIAL_TO_RUN} model (test f1 macro: {f1_test:.4f})")
fig.tight_layout()

cm_path = PLOTS_DIR / f"confusion_matrix_{TRIAL_TO_RUN}.png"
fig.savefig(cm_path, dpi=200, bbox_inches="tight")
print(f"Saved confusion matrix to {cm_path}")

plt.show()

In [ ]:
# Plot 2: validation score over trials (optimization history).
trials_df = study.trials_dataframe().sort_values("number").reset_index(drop=True)
trials_df["best_so_far"] = trials_df["value"].cummax()
best_trial_row = trials_df.loc[trials_df["value"].idxmax()]

fig, ax = plt.subplots(figsize=(10, 6))

# Best-so-far envelope as a dashed step line.
ax.step(
    trials_df["number"], trials_df["best_so_far"],
    where="post", color="tab:red", linestyle="--", linewidth=2,
    label="Best so far", zorder=2,
)

# Every trial, colored by its validation score.
scatter = ax.scatter(
    trials_df["number"], trials_df["value"],
    c=trials_df["value"], cmap="viridis",
    s=90, edgecolors="white", linewidths=0.8,
    label="Trial", zorder=3,
)

# Highlight the best trial.
ax.scatter(
    best_trial_row["number"], best_trial_row["value"],
    marker="*", s=400, color="tab:red", edgecolors="white", linewidths=0.8,
    label=f"Best trial (#{best_trial_row['number']:.0f})", zorder=4,
)

fig.colorbar(scatter, ax=ax, label="Validation f1 macro")
ax.set_xlabel("Trial number")
ax.set_ylabel("Validation f1 macro")
ax.set_title(f"Optuna optimization history — {TRIAL_TO_RUN}\nbest validation f1 macro: {study.best_value:.4f}")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.tight_layout()

history_path = PLOTS_DIR / f"optuna_history_{TRIAL_TO_RUN}.png"
fig.savefig(history_path, dpi=200, bbox_inches="tight")
print(f"Saved optimization history to {history_path}")

plt.show()